# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Lane 4 — CTR / Engagement Opportunity Scoring**

I want to identify pages that receive many Google Search impressions but relatively few clicks. The content editor needs to decide which pages should be reviewed first. For the selected pages, the editor can review elements such as the title, meta description, and alignment with search intent.

I frame this as a **scoring task**. Each page will receive a CTR opportunity score, and the pages will be prioritized using that score. I chose scoring because I want to see not only the order of the pages but also the numerical size of the opportunity associated with each page.

The output will be used by a content editor or SEO specialist. They will inspect the highest-scoring pages and decide whether their titles, meta descriptions, or intent alignment should be improved. The intended outcome is to make the search result more relevant and appealing and potentially increase its click-through rate.

A false positive would cause the editor to spend limited time reviewing a page that does not contain a meaningful opportunity. A false negative would cause a potentially valuable page to be missed, leading to missed clicks and, indirectly, possible customer or revenue opportunities.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



My proxy target is the gap between a page's observed CTR and the expected CTR of pages in the same position tier.

The proxy column will be called `ctr_opportunity_gap`:

**ctr_opportunity_gap = expected CTR for the position tier − observed page CTR**

A large positive value means that the page receives a lower CTR than other pages in a comparable search position. It may therefore deserve earlier editorial review.

This is a proxy rather than a true causal target. The dataset does not show whether CTR increased after an editor changed the page title or meta description. Therefore, I cannot claim that editing a high-scoring page will definitely increase its CTR. The proxy is only a directional signal for editorial decision support.

## 3. Success metric

*One metric you can defend. What number means 'good'?*


My primary success metric is **Precision@50**. After ranking pages by their CTR opportunity scores, this metric measures how many of the first 50 recommendations are confirmed by an editor as actionable CTR review opportunities.

I chose Precision@50 because the editor cannot inspect all 30,000 pages. The quality of the top of the review queue is therefore more important than general performance across the entire inventory.

For example, if an editor confirms that 35 of the first 50 recommendations are actionable opportunities:

**Precision@50 = 35 / 50 = 70%**

This metric evaluates the usefulness of the review queue, not the actual improvement in CTR after an edit. Measuring the real effect would require post-edit performance data.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

I use a minimum threshold of 500 impressions to reduce noisy CTR estimates based on very small samples. During data exploration, this threshold retained approximately 58.1% of the pages with usable position data. It provides a practical balance between keeping enough pages and requiring sufficient visibility. This is an analytical assumption and should be reviewed with the content team before operational use.

In [ ]:
import pandas as pd
data_url = "https://raw.githubusercontent.com/Ezgicode/FlyRAnk-AI/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_url)

print( "data inf", df.shape)
df.head()

df.columns.tolist()

data inf (30000, 44)


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [ ]:
lane_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier"
]

lane_df = df[lane_columns].copy()
lane_df.head()

,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier
0,content_304f48230142,3803,29,0.76,10.6,striking
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5


In [ ]:
valid_lane_df = lane_df[
    (lane_df["avg_position"] > 0)
    & (lane_df["impressions_90d"] >= 500)
    & (lane_df["position_tier"].notna())
].copy()

print("Initial number of pages:", len(lane_df))
print("Number of eligible pages:", len(valid_lane_df))
print(
    "Percentage of pages retained:",
    f"{len(valid_lane_df) / len(lane_df) * 100:.1f}%",
)

valid_lane_df.head()

Initial number of pages: 30000
Number of eligible pages: 16726
Percentage of pages retained: 55.8%


,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier
0,content_304f48230142,3803,29,0.76,10.6,striking
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5


In [ ]:
"Proxy"

expected_ctr_by_position = (
    valid_lane_df
    .groupby("position_tier", observed=True)["ctr"]
    .median()
)

valid_lane_df["expected_ctr_by_position"] = (
    valid_lane_df["position_tier"]
    .map(expected_ctr_by_position)
    .astype(float)
)

valid_lane_df["ctr_opportunity_gap"] = (
    valid_lane_df["expected_ctr_by_position"]
    - valid_lane_df["ctr"]
)

target_preview = valid_lane_df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "position_tier",
        "expected_ctr_by_position",
        "ctr_opportunity_gap",
    ]
]

target_preview.head(10)


,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,expected_ctr_by_position,ctr_opportunity_gap
0,content_304f48230142,3803,29,0.76,10.6,striking,0.17,-0.59
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5,0.09,0.04
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5,0.09,0.00
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1,0.24,-0.25
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5,0.09,-0.04
5,content_d4084a4bc775,3970,1,0.03,8.5,page_1,0.24,0.21
7,content_a63219c6e95a,1724,1,0.06,21.2,page_3_5,0.09,0.03
8,content_5e6c160719bc,32574,29,0.09,46.0,page_3_5,0.09,0.00
9,content_c27558df2b0c,1240,2,0.16,4.9,page_1,0.24,0.08
10,content_d8ee6cc6d642,20919,324,1.55,2.2,top_3,0.20,-1.35


In [ ]:
review_queue = valid_lane_df.sort_values(
    by=["ctr_opportunity_gap", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

# Merge content_type and main_intent from the original df
# because these columns were not included in valid_lane_df.
review_queue = pd.merge(
    review_queue,
    df[["content_id", "content_type", "main_intent"]],
    on="content_id",
    how="left"
)

review_queue[
    [
        "content_id",
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "position_tier",
        "expected_ctr_by_position",
        "ctr_opportunity_gap",
    ]
].head(20)

,content_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_tier,expected_ctr_by_position,ctr_opportunity_gap
0,content_c8e9d6ab9013,keyword article,informational,208678,0,0.0,9.7,page_1,0.24,0.24
1,content_f986bd514b6e,keyword article,NaN,22456,1,0.0,6.6,page_1,0.24,0.24
2,content_825a9788af8d,keyword article,informational,16786,0,0.0,5.6,page_1,0.24,0.24
3,content_8ba781dafa55,keyword article,informational,16156,0,0.0,9.0,page_1,0.24,0.24
4,content_5d5653c4eb4f,keyword article,informational,15101,0,0.0,5.7,page_1,0.24,0.24
5,content_847a841969a2,keyword article,transactional,14519,0,0.0,7.4,page_1,0.24,0.24
6,content_c82bc0c24241,keyword article,informational,13676,0,0.0,4.3,page_1,0.24,0.24
7,content_9983d31c53cb,keyword article,transactional,7737,0,0.0,5.5,page_1,0.24,0.24
8,content_d3aaf7d5f2fc,keyword article,transactional,7732,0,0.0,8.3,page_1,0.24,0.24
9,content_5195668f06db,keyword article,informational,6635,0,0.0,5.2,page_1,0.24,0.24


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could flag every page with a CTR below 1%. However, such a rule would ignore average search position and impression volume. A page in position 1 and a page in position 9 should not necessarily have the same expected CTR. A CTR calculated from only a few impressions may also be too noisy to support an editorial decision.

The review priority may depend on several interacting signals, including impressions, CTR, average position, position tier, content type, search intent, content age, and freshness. ML may be useful when these relationships are too complex, interconnected, or changeable to express reliably as one hand-written rule.

However, I will not assume that ML is automatically better. The transparent `ctr_opportunity_gap` score should serve as a baseline. A later model should be compared with this baseline using Precision@50. If the model does not produce a meaningful improvement, the simpler and more explainable scoring rule should be preferred.

The output is not an automatic editing decision. It is a decision-support queue that helps an editor choose which pages to inspect first. Low CTR does not prove that a title or meta description is poor, and a high opportunity score does not guarantee that an edit will increase clicks.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.